<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026/blob/main/02.Tokenization-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

## Colab에서 실행하는 경우, 다음과 같이 패키지 재설치가 필요하다:
* Jupyter Notebook에서는 `torchtext 0.17.0`을 사용했으나, 여기서는 `torch 2.3.0`에 맞는 `torchtext 0.18.0`을 사용하기로 한다.
* 최신 torch 버전에서는 `torchtext 0.18.0`을 지원하지 않는다. 따라서 Colab에 기본으로 설치된 torch를 uninstall하고 맞는 torch 버전으로 재설치해야 한다.
```
!pip uninstall torch torchvision torchaudio
import sys
!{sys.executable} -m pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0
!{sys.executable} -m pip install torchtext==0.18.0
```
`RESTART SESSION`해야 한다고 메시지를 받으면, RESTART 후 다음 셀부터 실행하면 된다.

In [4]:
!pip uninstall torch torchvision torchaudio
import sys
!{sys.executable} -m pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0
!{sys.executable} -m pip install torchtext==0.18.0

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Would remove:
    /usr/local/bin/torchfrtrace
    /usr/local/bin/torchrun
    /usr/local/lib/python3.12/dist-packages/functorch/*
    /usr/local/lib/python3.12/dist-packages/torch-2.10.0+cpu.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torch/*
    /usr/local/lib/python3.12/dist-packages/torchgen/*
Proceed (Y/n)? y
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchvision-0.25.0+cpu.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libjpeg.4af9affd.so.8
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libpng16.c2edc9e1.so.16
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libsharpyuv.994c9d2c.so.0
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libwebp.87a45f40.so.7
    /usr/local/lib/python3.1

<span style="font-size:2.5em">2. 텍스트 전처리 및 토큰화 (1)</span>

[참조]
- [Natural Language Processing with Python (NLTK Books)](https://www.nltk.org/book/)

#### <span style="color: crimson">[Remark🐞]</span> 쥬피터 셀에서 파이썬 패키지 설치시 권장되는 방법

```
import sys

!conda install --yes --prefix {sys.prefix} package_to_install
# 또는
!{sys.executable} -m pip install package_to_install
```


[참조] [Installing Python Packages from a Jupyter Notebook](https://jakevdp.github.io/blog/2017/12/05/installing-python-packages-from-jupyter/)

# 2.1 단어 토큰화(Word Tokenization)

### Tokenizer 비교 - 선택의 문제

In [7]:
sentence = "Mr. Jone's house isn't far from here, New York."

#### 띄어쓰기(space)로 토큰화
- string 메서드 `.split()` 사용

In [8]:
sentence.split()

['Mr.', "Jone's", 'house', "isn't", 'far', 'from', 'here,', 'New', 'York.']

In [9]:
text = "멀리 가지 마. 김 박사님 너를 보러 오실거야. 네가 없으면 당황스러울거야."
text.split()

['멀리', '가지', '마.', '김', '박사님', '너를', '보러', '오실거야.', '네가', '없으면', '당황스러울거야.']

#### NLTK `word_tokenize`

In [10]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [11]:
from nltk.tokenize import word_tokenize
print(word_tokenize(sentence))

['Mr.', 'Jone', "'s", 'house', 'is', "n't", 'far', 'from', 'here', ',', 'New', 'York', '.']


#### NLTK `WordPunctTokenizer`

In [13]:
from nltk.tokenize import WordPunctTokenizer
print(WordPunctTokenizer().tokenize(sentence))

['Mr', '.', 'Jone', "'", 's', 'house', 'isn', "'", 't', 'far', 'from', 'here', ',', 'New', 'York', '.']


#### NLTK `TreebankWordTokenizer`
- Penn Treebank Tokenizer
- 정규표현식을 사용하여 Penn Treebank 스타일의 토큰화를 제공하는 도구

In [14]:
from nltk.tokenize import TreebankWordTokenizer
tokenizer = TreebankWordTokenizer()
print(tokenizer.tokenize(sentence))

['Mr.', 'Jone', "'s", 'house', 'is', "n't", 'far', 'from', 'here', ',', 'New', 'York', '.']


#### PyTorch torchtext가 제공하는 tokenizer 중 `basic_english`
- torchtext가 제공하는 tokenizer: `basic_english`, `spacy`, `moses`, `toktok`, `revtok`, `subword`

In [15]:
### Colab을 사용하는 경우 다음 코드를 커멘트 처리하여 건너 뛰도록 한다.
import sys
!{sys.executable} -m pip install torchtext

In [16]:
from torchtext.data import get_tokenizer
print(get_tokenizer("basic_english")(sentence))

['mr', '.', 'jone', "'", 's', 'house', 'isn', "'", 't', 'far', 'from', 'here', ',', 'new', 'york', '.']


#### TensorFlow `text_to_word_sequence`

In [17]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence
print(text_to_word_sequence(sentence))

['mr', "jone's", 'house', "isn't", 'far', 'from', 'here', 'new', 'york']


- 텐서플로우 tensorflow.keras.preprocessing.text의 `text_to_word_sequence` 나 `Tokenizer`는 `tf.Tensor`를 지원하지 않음
- 권장 방법 참고: [텍스트 로드 하기](https://www.tensorflow.org/tutorials/load_data/text?hl=ko)

# 2.2. 문장 분할(Sentence Segmentation)

#### NLTK `sent_tokenize`

In [18]:
from nltk.tokenize import sent_tokenize

text = "Don't go far. Dr. Kim is coming to see you. I'd be embarrassed without you."
print(sent_tokenize(text))

["Don't go far.", 'Dr. Kim is coming to see you.', "I'd be embarrassed without you."]


In [19]:
text = "멀리 가지 마. 김 박사님 너를 보러 오실거야. 네가 없으면 당황스러울거야."
print(sent_tokenize(text))

['멀리 가지 마.', '김 박사님 너를 보러 오실거야.', '네가 없으면 당황스러울거야.']


In [20]:
text = "Go to the IP 192.168.56.31 server, save the log file, and send the results to aaa@gmail.com. After that, let's go have lunch."
print(sent_tokenize(text))

['Go to the IP 192.168.56.31 server, save the log file, and send the results to aaa@gmail.com.', "After that, let's go have lunch."]


In [21]:
text = "IP 192.168.56.31 서버에 들어가서 로그 파일 저장해서 aaa@gmail.com 로 결과 좀 보내줘. 그 후 점심 먹으러 가자."
print(sent_tokenize(text))

['IP 192.168.56.31 서버에 들어가서 로그 파일 저장해서 aaa@gmail.com 로 결과 좀 보내줘.', '그 후 점심 먹으러 가자.']


#### KSS `split_sentences`
- KSS(Korean Sentence Splitter) - 박상길

In [22]:
import sys
!{sys.executable} -m pip install kss

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 60.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.8/468.8 kB 21.6 MB/s eta 0:00:00
  Created wheel for kss: filename=kss

In [23]:
import kss

text = "멀리 가지 마. 김 박사님 너를 보러 오실거야. 네가 없으면 당황스러울거야."
print(kss.split_sentences(text))

For your information, Kss also supports mecab backend.
We recommend you to install mecab or konlpy.tag.Mecab for faster execution of Kss.
Please refer to following web sites for details:
- mecab: https://github.com/hyunwoongko/python-mecab-kor
- konlpy.tag.Mecab: https://konlpy.org/en/latest/api/konlpy.tag/#mecab-class



['멀리 가지 마.', '김 박사님 너를 보러 오실거야.', '네가 없으면 당황스러울거야.']


/usr/local/lib/python3.12/dist-packages/pecab/_tokenizer.py:265: RuntimeWarning: overflow encountered in scalar add
  from_pos_data.costs[idx]


- [MS Windows 에 MeCab 설치하는 방법](https://wonhwa.tistory.com/49)
- [Windows Python 3.x에 MeCab 설치법](https://cleancode-ws.tistory.com/97)

In [24]:
text = "IP 192.168.56.31 서버에 들어가서 로그 파일 저장해서 aaa@gmail.com 로 결과 좀 보내줘. 그 후 점심 먹으러 가자."
print(kss.split_sentences(text))

['IP 192.168.56.31 서버에 들어가서 로그 파일 저장해서 aaa@gmail.com 로 결과 좀 보내줘.', '그 후 점심 먹으러 가자.']


In [25]:
text = "Go to the IP 192.168.56.31 server, save the log file, and send the results to aaa@gmail.com. After that, let's go have lunch."
print(kss.split_sentences(text))

["Go to the IP 192.168.56.31 server, save the log file, and send the results to aaa@gmail.com. After that, let's go have lunch."]


- 문장 분할 실패!

# 2.3 토큰화 및 POS Tagging

### NLTK

In [26]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [27]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag

In [28]:
text = "Don't go far. Dr. Kim is coming to see you. I'd be embarrassed without you."
tokenized_sentence = word_tokenize(text)

print('단어 토큰화 :',tokenized_sentence)
print('품사 태깅 :',pos_tag(tokenized_sentence))

단어 토큰화 : ['Do', "n't", 'go', 'far', '.', 'Dr.', 'Kim', 'is', 'coming', 'to', 'see', 'you', '.', 'I', "'d", 'be', 'embarrassed', 'without', 'you', '.']
품사 태깅 : [('Do', 'VBP'), ("n't", 'RB'), ('go', 'VB'), ('far', 'RB'), ('.', '.'), ('Dr.', 'NNP'), ('Kim', 'NNP'), ('is', 'VBZ'), ('coming', 'VBG'), ('to', 'TO'), ('see', 'VB'), ('you', 'PRP'), ('.', '.'), ('I', 'PRP'), ("'d", 'MD'), ('be', 'VB'), ('embarrassed', 'VBN'), ('without', 'IN'), ('you', 'PRP'), ('.', '.')]


In [29]:
sentences = sent_tokenize(text)
for sent in sentences:
    tokenized = word_tokenize(sent)
    print(tokenized)
    print(pos_tag(tokenized))

['Do', "n't", 'go', 'far', '.']
[('Do', 'VBP'), ("n't", 'RB'), ('go', 'VB'), ('far', 'RB'), ('.', '.')]
['Dr.', 'Kim', 'is', 'coming', 'to', 'see', 'you', '.']
[('Dr.', 'NNP'), ('Kim', 'NNP'), ('is', 'VBZ'), ('coming', 'VBG'), ('to', 'TO'), ('see', 'VB'), ('you', 'PRP'), ('.', '.')]
['I', "'d", 'be', 'embarrassed', 'without', 'you', '.']
[('I', 'PRP'), ("'d", 'MD'), ('be', 'VB'), ('embarrassed', 'VBN'), ('without', 'IN'), ('you', 'PRP'), ('.', '.')]


### KoNLPy
- 지원하는 형태소 분석기: Okt(Open Korea Text), 메캅(Mecab), 코모란(Komoran), 한나눔(Hannanum), 꼬꼬마(Kkma)

In [30]:
import sys
!{sys.executable} -m pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 27.4 MB/s eta 0:00:00


In [31]:
from konlpy.tag import Okt
from konlpy.tag import Kkma

In [32]:
okt = Okt()
kkma = Kkma()
text = "3월에 대학생들이 강한 이유는? 개강해서!"

#### OKT(Open Korea Text)

In [33]:
print('OKT 형태소 분석 :',okt.morphs(text))
print('OKT 품사 태깅 :',okt.pos(text))
print('OKT 명사 추출 :',okt.nouns(text))

OKT 형태소 분석 : ['3월', '에', '대학생', '들', '이', '강한', '이유', '는', '?', '개강', '해서', '!']
OKT 품사 태깅 : [('3월', 'Number'), ('에', 'Foreign'), ('대학생', 'Noun'), ('들', 'Suffix'), ('이', 'Josa'), ('강한', 'Adjective'), ('이유', 'Noun'), ('는', 'Josa'), ('?', 'Punctuation'), ('개강', 'Noun'), ('해서', 'Verb'), ('!', 'Punctuation')]
OKT 명사 추출 : ['대학생', '이유', '개강']


#### 꼬꼬마(kma)

In [34]:
print('꼬꼬마 형태소 분석 :',kkma.morphs(text))
print('꼬꼬마 품사 태깅 :',kkma.pos(text))
print('꼬꼬마 명사 추출 :',kkma.nouns(text))

꼬꼬마 형태소 분석 : ['3', '월', '에', '대학생', '들', '이', '강하', 'ㄴ', '이유', '는', '?', '개강', '하', '어서', '!']
꼬꼬마 품사 태깅 : [('3', 'NR'), ('월', 'NNM'), ('에', 'JKM'), ('대학생', 'NNG'), ('들', 'XSN'), ('이', 'JKS'), ('강하', 'VV'), ('ㄴ', 'ETD'), ('이유', 'NNG'), ('는', 'JX'), ('?', 'SF'), ('개강', 'NNG'), ('하', 'XSV'), ('어서', 'ECD'), ('!', 'SF')]
꼬꼬마 명사 추출 : ['3', '3월', '월', '대학생', '이유', '개강']


# 2.4 불용어(Stopwords) 제거

### NLTK `stopwords`

In [35]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [36]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from konlpy.tag import Okt

In [37]:
stop_words_list = stopwords.words('english')
print(len(stop_words_list))
print(stop_words_list)

198
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 

In [38]:
text ="""DENNIS: Listen, strange women lying in ponds distributing swords
is no basis for a system of government.  Supreme executive power derives from
a mandate from the masses, not from some farcical aquatic ceremony."""
word_tokens = word_tokenize(text)
result = []
for word in word_tokens:
    if word not in stop_words_list:
        result.append(word)
print(word_tokens)
print(result)

['DENNIS', ':', 'Listen', ',', 'strange', 'women', 'lying', 'in', 'ponds', 'distributing', 'swords', 'is', 'no', 'basis', 'for', 'a', 'system', 'of', 'government', '.', 'Supreme', 'executive', 'power', 'derives', 'from', 'a', 'mandate', 'from', 'the', 'masses', ',', 'not', 'from', 'some', 'farcical', 'aquatic', 'ceremony', '.']
['DENNIS', ':', 'Listen', ',', 'strange', 'women', 'lying', 'ponds', 'distributing', 'swords', 'basis', 'system', 'government', '.', 'Supreme', 'executive', 'power', 'derives', 'mandate', 'masses', ',', 'farcical', 'aquatic', 'ceremony', '.']


### KoNLPy

In [39]:
okt = Okt()
text = """고기를 아무렇게나 구우려고 하면 안 돼. 고기라고 다 같은 게 아니거든.
예컨대 삼겹살을 구울 때는 중요한 게 있지."""
stop_words = "\n 를 아무렇게나 고 안 돼 같은 게 구울 때 는"
stop_words_list = set(stop_words.split(' '))
word_tokens = okt.morphs(text)

result = []
for word in word_tokens:
    if word not in stop_words_list:
        result.append(word)

print('불용어 제거 전 :',word_tokens)
print('불용어 제거 후 :',result)

불용어 제거 전 : ['고기', '를', '아무렇게나', '구', '우려', '고', '하면', '안', '돼', '.', '고기', '라고', '다', '같은', '게', '아니거든', '.', '\n', '예컨대', '삼겹살', '을', '구울', '때', '는', '중요한', '게', '있지', '.']
불용어 제거 후 : ['고기', '구', '우려', '하면', '.', '고기', '라고', '다', '아니거든', '.', '예컨대', '삼겹살', '을', '중요한', '있지', '.']


### `many-stop-words`

In [40]:
import sys
!{sys.executable} -m pip install many-stop-words

  Preparing metadata (setup.py) ... done
  Created wheel for many-stop-words: filename=many_stop_words-0.2.2-py3-none-any.whl size=26220 sha256=2e309afa273828430ddc07b03a3949c2ec550ddcdf6842445010f4093f6905c9
  Stored in directory: /root/.cache/pip/wheels/4f/d0/78/f435a7b16827117fa40f938d9bc50b709276221460fec21a5a
Successfully built many-stop-words


In [41]:
from many_stop_words import get_stop_words, available_languages
import many_stop_words
print("지원 언어 : ", len(many_stop_words.available_languages))
print(many_stop_words.available_languages)

지원 언어 :  22
['ar', 'ca', 'cs', 'de', 'el', 'en', 'es', 'fi', 'fr', 'hu', 'it', 'ja', 'kr', 'nl', 'no', 'pl', 'pt', 'ru', 'sk', 'sv', 'tr', 'zh']


In [42]:
print(len(many_stop_words.get_stop_words('kr')))
print(many_stop_words.get_stop_words('kr'))

595
{'헉헉', '자마자', '그러면', '인 듯하다', '본대로', '더구나', '까지도', '하느니', '무슨', '의해되다', '습니까', '한켠으로는', '가령', '전자', '아니', '그때', '하지만', '비하면', '휴', '바꿔 말하면', '어떤것들', '퉤', '그런데', '저희', '해서는 안된다', '바꾸어말하자면', '않기 위해서', '에 달려 있다', '어', '근거하여', '만일', '등', '어찌됏든', '일곱', '할수있다', '자신', '하는 김에', '이외에도', '함께', '아하', '줄은모른다', '여덟', '주저하지 않고', '하더라도', '각자', '야', '요만큼', '만은 아니다', '첫번째로', '이와 같은', '시각', '잠깐', '이 되다', '이번', '할 지경이다', '아이야', '우에 종합한것과같이', '하면된다', '한다면 몰라도', '곧', '할 따름이다', '혹은', '그에 따르는', '대해 말하자면', '즉시', '하도록하다', '외에도', '좍좍', '반대로', '헐떡헐떡', '흐흐', '그', '만약에', '와', '과', '등등', '시간', '따위', '이지만', '기대여', '누구', '그만이다', '저쪽', '타인', '혹시', '바꾸어말하면', '약간', '일반적으로', '이천팔', '무렵', '하면서', '네', '의', '위에서 서술한바와같이', '그러니', '지말고', '몇', '말하자면', '더라도', '하도다', '할 줄 안다', '뿐이다', '그리고', '만약', '것들', '이와 반대로', '와아', '할망정', '로써', '예컨대', '하겠는가', '비슷하다', '잇따라', '한적이있다', '나머지는', '륙', '월', '그렇지만', '하구나', '임에 틀림없다', '마음대로', '아이고', '어느때', '하지 않도록', '비길수 없다', '에 있다', '한데', '구토하다', '윙윙', '하는것이 낫다', '차라리', '어느해', '하기 위하여', '메쓰겁다', '

# 2.5 Normalizing Text

Normalization은 다음과 같은 작업을 포함한다:
- 텍스트를 모두 lowercase로 바꾸는 일
- stemming을 통해 접사들을 떼어버리거나,
- lemmatization을 통해 단어를 사전 표제어로 나타나는 형태로 바꾸는 작업들

In [43]:
import nltk
from nltk import word_tokenize

text ="""DENNIS: Listen, strange women lying in ponds distributing swords
is no basis for a system of government.  Supreme executive power derives from
a mandate from the masses, not from some farcical aquatic ceremony."""
tokens = word_tokenize(text)
print(tokens)

['DENNIS', ':', 'Listen', ',', 'strange', 'women', 'lying', 'in', 'ponds', 'distributing', 'swords', 'is', 'no', 'basis', 'for', 'a', 'system', 'of', 'government', '.', 'Supreme', 'executive', 'power', 'derives', 'from', 'a', 'mandate', 'from', 'the', 'masses', ',', 'not', 'from', 'some', 'farcical', 'aquatic', 'ceremony', '.']


## Stemming (어간 추출)
-  RE을 사용하여 수작업으로 할 수도 있으나, NLTK 내장 stemmer는 불규칙적인 경우들을 처리할 수 있으므로 더 효율적이다.
- NLTK 내장 stemmer:
  - Porter stemmer
  - Lancaster stemmer

In [ ]:
porter = nltk.PorterStemmer()
print([porter.stem(t) for t in tokens])

['denni', ':', 'listen', ',', 'strang', 'women', 'lie', 'in', 'pond', 'distribut', 'sword', 'is', 'no', 'basi', 'for', 'a', 'system', 'of', 'govern', '.', 'suprem', 'execut', 'power', 'deriv', 'from', 'a', 'mandat', 'from', 'the', 'mass', ',', 'not', 'from', 'some', 'farcic', 'aquat', 'ceremoni', '.']


In [ ]:
lancaster = nltk.LancasterStemmer()
print([lancaster.stem(t) for t in tokens])

['den', ':', 'list', ',', 'strange', 'wom', 'lying', 'in', 'pond', 'distribut', 'sword', 'is', 'no', 'bas', 'for', 'a', 'system', 'of', 'govern', '.', 'suprem', 'execut', 'pow', 'der', 'from', 'a', 'mand', 'from', 'the', 'mass', ',', 'not', 'from', 'som', 'farc', 'aqu', 'ceremony', '.']


- Stemming은 잘 정의된 과정이 아니며 목적에 맞는 것을 선택해 사용한다
- 텍스트 인텍싱 등 일반적인 NLP 작업에 있어 Porter stemmer 성능이 Lancaster stemmer 보다 좋음

#### Stemmer를 사용한 텍스트 인덱싱
- `concordance` view는 주어진 단어가 텍스트 내에서 출현하는 것을 주변 단어들(context)과 함께 보여준다.

In [ ]:
class IndexedText(object):

    def __init__(self, stemmer, text):
        self._text = text
        self._stemmer = stemmer
        self._index = nltk.Index((self._stem(word), i) # nltk.Index source code: https://tedboy.github.io/nlps/_modules/nltk/util.html#Index
                                 for (i, word) in enumerate(text))
    def concordance(self, word, width=40):
        key = self._stem(word)
        wc = int(width/4)                # words of context, 왜 4로 나눴을까?
        for i in self._index[key]:
            lcontext = ' '.join(self._text[i-wc:i])
            rcontext = ' '.join(self._text[i:i+wc])
            ldisplay = '{:>{width}}'.format(lcontext[-width:], width=width)  # ':>{width}' width 길이 만큼 표시하되 오른쪽 맞춤한다
            rdisplay = '{:{width}}'.format(rcontext[:width], width=width)
            print(ldisplay, rdisplay)

    def _stem(self, word):
        return self._stemmer.stem(word).lower()

In [ ]:
import nltk
nltk.download('webtext')

[nltk_data] Downloading package webtext to
[nltk_data]     C:\Users\kikim\AppData\Roaming\nltk_data...
[nltk_data]   Package webtext is already up-to-date!


True

In [ ]:
porter = nltk.PorterStemmer()
grail = nltk.corpus.webtext.words('grail.txt')
text = IndexedText(porter, grail)
text.concordance('lie')

r king ! DENNIS : Listen , strange women lying in ponds distributing swords is no
 beat a very brave retreat . ROBIN : All lies ! MINSTREL : [ singing ] Bravest of
       Nay . Nay . Come . Come . You may lie here . Oh , but you are wounded !   
doctors immediately ! No , no , please ! Lie down . [ clap clap ] PIGLET : Well  
ere is much danger , for beyond the cave lies the Gorge of Eternal Peril , which 
   you . Oh ... TIM : To the north there lies a cave -- the cave of Caerbannog --
h it and lived ! Bones of full fifty men lie strewn about its lair . So , brave k
not stop our fight ' til each one of you lies dead , and the Holy Grail returns t


## Lemmatization (표제어 추출)

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kikim\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kikim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
text ="""Reading stories is a great way to improve your vocabulary
and we have lots of great stories for you to watch."""
tokens = word_tokenize(text)

In [ ]:
lemmatizer = nltk.WordNetLemmatizer()
print(tokens)
print([lemmatizer.lemmatize(t) for t in tokens])

['Reading', 'stories', 'is', 'a', 'great', 'way', 'to', 'improve', 'your', 'vocabulary', 'and', 'we', 'have', 'lots', 'of', 'great', 'stories', 'for', 'you', 'to', 'watch', '.']
['Reading', 'story', 'is', 'a', 'great', 'way', 'to', 'improve', 'your', 'vocabulary', 'and', 'we', 'have', 'lot', 'of', 'great', 'story', 'for', 'you', 'to', 'watch', '.']


In [ ]:
words = ['policy', 'doing', 'organization', 'have', 'going', 'love', 'lives', 'fly', 'dies', 'watched', 'has', 'starting']
print(words)
print([lemmatizer.lemmatize(t) for t in words])

['policy', 'doing', 'organization', 'have', 'going', 'love', 'lives', 'fly', 'dies', 'watched', 'has', 'starting']
['policy', 'doing', 'organization', 'have', 'going', 'love', 'life', 'fly', 'dy', 'watched', 'ha', 'starting']


- WordNet lemmatizer는 접사를 잘라낸 단어가 사전에 있을 때만 접사를 자른다
- 추가 처리 때문에 stemmer 보다 느리다
- 'lying'을 처리하지 않았지만 'women'은 'woman'으로 바꿨다.

# 2.6 정규표현식 (Regular Expressions)

- regular expression을 표시할 때 «ed$» 처럼 «정규식» 기호를 사용할 것임.
- Python에서의 regular expression은 `re` library를 사용한다

In [45]:
import nltk
nltk.download('words')

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [46]:
import re
wordlist = [w for w in nltk.corpus.words.words('en') if w.islower()]

## 1) 기본 Meta-characters 사용하기
- 'ed'로 끝나는 단어 찾기
- `$`: end of a word

In [ ]:
[w for w in wordlist if re.search('ed$', w)]

['abaissed',
 'abandoned',
 'abased',
 'abashed',
 'abatised',
 'abed',
 'aborted',
 'abridged',
 'abscessed',
 'absconded',
 'absorbed',
 'abstracted',
 'abstricted',
 'accelerated',
 'accepted',
 'accidented',
 'accoladed',
 'accolated',
 'accomplished',
 'accosted',
 'accredited',
 'accursed',
 'accused',
 'accustomed',
 'acetated',
 'acheweed',
 'aciculated',
 'aciliated',
 'acknowledged',
 'acorned',
 'acquainted',
 'acquired',
 'acquisited',
 'acred',
 'aculeated',
 'addebted',
 'added',
 'addicted',
 'addlebrained',
 'addleheaded',
 'addlepated',
 'addorsed',
 'adempted',
 'adfected',
 'adjoined',
 'admired',
 'admitted',
 'adnexed',
 'adopted',
 'adossed',
 'adreamed',
 'adscripted',
 'aduncated',
 'advanced',
 'advised',
 'aeried',
 'aethered',
 'afeared',
 'affected',
 'affectioned',
 'affined',
 'afflicted',
 'affricated',
 'affrighted',
 'affronted',
 'aforenamed',
 'afterfeed',
 'aftershafted',
 'afterthoughted',
 'afterwitted',
 'agazed',
 'aged',
 'agglomerated',
 'aggri

- 8 글자 단어 중 3번째가 j 이고 6번째가 t인 단어 찾기
- `.`: any single character
- `^`: start of a string

In [50]:
[w for w in wordlist if re.search('..j..t..$', w)]

['abjectly',
 'adjuster',
 'coprojector',
 'dejected',
 'dejectly',
 'injector',
 'interjector',
 'majestic',
 'maladjusted',
 'microprojector',
 'munjistin',
 'objectee',
 'objector',
 'projector',
 'readjuster',
 'rejecter',
 'rejector',
 'subjected',
 'unadjusted',
 'undejected',
 'unejected',
 'uninjected',
 'uninterjected',
 'unjilted',
 'unjolted',
 'unjustly',
 'unmajestic',
 'unobjected',
 'unprojected',
 'unsubjected']

#### <span style="color:crimson">[연습😉1]</span> 위의 예에서 `^` 기호를 없앤다면 결과는?

- `?`: 앞의 글자는 option이라는 의미. (예) «^e-?mail$» 는 'email'과 'e-mail' 모두 일치한다.
- 다음과 같은 코드로 이러한 단어의 총 빈도수를 셀수 있다.
```python
sum(1 for w in text if re.search('^e-?mail$', w))
```

### 범위(range) 및 클로저(closure)

<img src="https://github.com/kikim6114/nlp2026/blob/main/images/9key_pad.png?raw=1" width="200" style="margin-left: auto; margin-right: auto">
<p style="text-align: center;">키패드를 사용한 텍스트 입력</p>

- **textonyms**: 같은 keystroke 순서로 입력되는 둘 또는 그 이상의 단어들
- (예) hole과 gold의 키 순서는 4653
- 이와 같은 키 순서의 다른 단어를 찾아보자.

In [ ]:
[w for w in wordlist if re.search('^[ghi][mno][jlk][def]$', w)]

['gold', 'golf', 'hold', 'hole']

#### <span style="color:crimson">[연습😉2]</span> 숫자 패드의 일부분만 사용하는 단어인 "finger-twister"를 찾아보자.

- 예를 들어, «^\[ghijklmno\]\+\\$» 또는 더 축약해서 «^[g-o]+\\$»는 가운데 행의 4, 5, 6 키만 사용하는 단어와 match할 것이다. «^[a-fj-o]+$» 는 2, 3, 5, 6 키들만 사용하는 단어들과 매치된다. 추가로, 여기서 `-` 와 `+`는 무슨 의미인가?

- `+`: 바로 앞의 아이템이 하나 또는 그 이상 반복됨
- `*`: 바로 앞의 아이템이 제로 또는 그 이상 반복됨
- 이 두 기호를 <span style="color:blue">**Kleene closures**</span> 또는 단순히 <span style="color: blue">**closures**</span> 라고 함.

In [52]:
import nltk
nltk.download('nps_chat')

[nltk_data] Downloading package nps_chat to /root/nltk_data...
[nltk_data]   Unzipping corpora/nps_chat.zip.


True

In [53]:
chat_words = sorted(set(w for w in nltk.corpus.nps_chat.words()))
chat_words

['',
 '!',
 '!!',
 '!!!',
 '!!!!',
 '!!!!!',
 '!!!!!!',
 '!!!!!!!',
 '!!!!!!!!',
 '!!!!!!!!!',
 '!!!!!!!!!!',
 '!!!!!!!!!!!',
 '!!!!!!!!!!!!!',
 '!!!!!!!!!!!!!!!!',
 '!!!!!!!!!!!!!!!!!!!!!!',
 '!!!!!!!!!!!!!!!!!!!!!!!',
 '!!!!!!!!!!!!!!!!!!!!!!!!!!!',
 '!!!!!!!!!!!!!!!!!!!!!!!!!!!!',
 '!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!',
 '!!!!!!.',
 '!!!!!.',
 '!!!!....',
 '!!!.',
 '!!.',
 '!!...',
 '!.',
 '!...',
 '!=',
 '!?',
 '!??',
 '!???',
 '"',
 '"...',
 '"?',
 '"s',
 '#',
 '###',
 '####',
 '#14-19teens',
 '#40sPlus',
 '#prideIsland',
 '#prideisland',
 '#talkcity-20s',
 '#talkcity_adults',
 '$',
 '$$',
 '$27',
 '&',
 '&^',
 "'",
 "''",
 "'.",
 "'d",
 "'ello",
 "'ll",
 "'m",
 "'n'",
 "'re",
 "'s",
 "'ve",
 '(',
 '( o Y o )',
 '(((',
 '((((',
 '(((((',
 '((((((',
 '(((((((',
 '((((((((',
 '(((((((((',
 '((((((((((',
 '(((((((((((',
 '((((((((((((',
 '(((((((((((((',
 '((((((((((((((',
 '(((((((((((((((',
 '(((((((((((((((((',
 '((((((((((((((((((',
 '((((((((((((((((((((',
 '(

In [ ]:
[w for w in chat_words if re.search('^m+i+n+e+$', w)]

['miiiiiiiiiiiiinnnnnnnnnnneeeeeeeeee',
 'miiiiiinnnnnnnnnneeeeeeee',
 'mine',
 'mmmmmmmmiiiiiiiiinnnnnnnnneeeeeeee']

In [54]:
[w for w in chat_words if re.search('^[ha]+$', w)]

['a',
 'aaaaaaaaaaaaaaaaa',
 'aaahhhh',
 'ah',
 'ahah',
 'ahahah',
 'ahh',
 'ahhahahaha',
 'ahhh',
 'ahhhh',
 'ahhhhhh',
 'ahhhhhhhhhhhhhh',
 'h',
 'ha',
 'haaa',
 'hah',
 'haha',
 'hahaaa',
 'hahah',
 'hahaha',
 'hahahaa',
 'hahahah',
 'hahahaha',
 'hahahahaaa',
 'hahahahahaha',
 'hahahahahahaha',
 'hahahahahahahahahahahahahahahaha',
 'hahahhahah',
 'hahhahahaha']

`^`가 square bracket 내의 첫글자에 있는 경우
- (예) «[^aeiouAEIOU]» : 모음이 아닌 모든 글자
- (예) «^[^aeiouAEIOU]+\\$» : 모음이 아닌 글자로만 된 단어

`\`, `{}`, `()`, `|` 의 사용법

In [56]:
import nltk
nltk.download('treebank')

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.


True

In [57]:
wsj = sorted(set(nltk.corpus.treebank.words()))

In [60]:
[w for w in wsj if re.search('^[0-9]+\.[0-9]+$', w)]

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_312/2475292011.py:1: SyntaxWarning: invalid escape sequence '\.'
  [w for w in wsj if re.search('^[0-9]+\.[0-9]+$', w)]


['0.0085',
 '0.05',
 '0.1',
 '0.16',
 '0.2',
 '0.25',
 '0.28',
 '0.3',
 '0.4',
 '0.5',
 '0.50',
 '0.54',
 '0.56',
 '0.60',
 '0.7',
 '0.82',
 '0.84',
 '0.9',
 '0.95',
 '0.99',
 '1.01',
 '1.1',
 '1.125',
 '1.14',
 '1.1650',
 '1.17',
 '1.18',
 '1.19',
 '1.2',
 '1.20',
 '1.24',
 '1.25',
 '1.26',
 '1.28',
 '1.35',
 '1.39',
 '1.4',
 '1.457',
 '1.46',
 '1.49',
 '1.5',
 '1.50',
 '1.55',
 '1.56',
 '1.5755',
 '1.5805',
 '1.6',
 '1.61',
 '1.637',
 '1.64',
 '1.65',
 '1.7',
 '1.75',
 '1.76',
 '1.8',
 '1.82',
 '1.8415',
 '1.85',
 '1.8500',
 '1.9',
 '1.916',
 '1.92',
 '10.19',
 '10.2',
 '10.5',
 '107.03',
 '107.9',
 '109.73',
 '11.10',
 '11.5',
 '11.57',
 '11.6',
 '11.72',
 '11.95',
 '112.9',
 '113.2',
 '116.3',
 '116.4',
 '116.7',
 '116.9',
 '118.6',
 '12.09',
 '12.5',
 '12.52',
 '12.68',
 '12.7',
 '12.82',
 '12.97',
 '120.7',
 '1206.26',
 '121.6',
 '126.1',
 '126.15',
 '127.03',
 '129.91',
 '13.1',
 '13.15',
 '13.5',
 '13.50',
 '13.625',
 '13.65',
 '13.73',
 '13.8',
 '13.90',
 '130.6',
 '130.7',
 '

In [62]:
[w for w in wsj if re.search('^[A-Z]+\$$', w)]

<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_312/1759786652.py:1: SyntaxWarning: invalid escape sequence '\$'
  [w for w in wsj if re.search('^[A-Z]+\$$', w)]


['C$', 'US$']

In [61]:
[w for w in wsj if re.search('^[0-9]{4}$', w)]

['1614',
 '1637',
 '1787',
 '1901',
 '1903',
 '1917',
 '1925',
 '1929',
 '1933',
 '1934',
 '1948',
 '1953',
 '1955',
 '1956',
 '1961',
 '1965',
 '1966',
 '1967',
 '1968',
 '1969',
 '1970',
 '1971',
 '1972',
 '1973',
 '1975',
 '1976',
 '1977',
 '1979',
 '1980',
 '1981',
 '1982',
 '1983',
 '1984',
 '1985',
 '1986',
 '1987',
 '1988',
 '1989',
 '1990',
 '1991',
 '1992',
 '1993',
 '1994',
 '1995',
 '1996',
 '1997',
 '1998',
 '1999',
 '2000',
 '2005',
 '2009',
 '2017',
 '2019',
 '2029',
 '3057',
 '8300']

In [63]:
[w for w in wsj if re.search('^[0-9]+-[a-z]{3,5}$', w)]

['10-day',
 '10-lap',
 '10-year',
 '100-share',
 '12-point',
 '12-year',
 '14-hour',
 '15-day',
 '150-point',
 '190-point',
 '20-point',
 '20-stock',
 '21-month',
 '237-seat',
 '240-page',
 '27-year',
 '30-day',
 '30-point',
 '30-share',
 '30-year',
 '300-day',
 '36-day',
 '36-store',
 '42-year',
 '50-state',
 '500-stock',
 '52-week',
 '69-point',
 '84-month',
 '87-store',
 '90-day']

In [ ]:
[w for w in wsj if re.search('^[a-z]{5,}-[a-z]{2,3}-[a-z]{,6}$', w)]

['black-and-white',
 'bread-and-butter',
 'father-in-law',
 'machine-gun-toting',
 'savings-and-loan']

In [64]:
[w for w in wsj if re.search('(ed|ing)$', w)]

['62%-owned',
 'Absorbed',
 'According',
 'Adopting',
 'Advanced',
 'Advancing',
 'Alfred',
 'Allied',
 'Annualized',
 'Anything',
 'Arbitrage-related',
 'Arbitraging',
 'Asked',
 'Assuming',
 'Atlanta-based',
 'Baking',
 'Banking',
 'Beginning',
 'Beijing',
 'Being',
 'Bermuda-based',
 'Betting',
 'Boeing',
 'Broadcasting',
 'Bucking',
 'Buying',
 'Calif.-based',
 'Change-ringing',
 'Citing',
 'Concerned',
 'Confronted',
 'Conn.based',
 'Consolidated',
 'Continued',
 'Continuing',
 'Declining',
 'Defending',
 'Depending',
 'Designated',
 'Determining',
 'Developed',
 'Died',
 'During',
 'Encouraged',
 'Encouraging',
 'English-speaking',
 'Estimated',
 'Everything',
 'Excluding',
 'Exxon-owned',
 'Faulding',
 'Fed',
 'Feeding',
 'Filling',
 'Filmed',
 'Financing',
 'Following',
 'Founded',
 'Fracturing',
 'Francisco-based',
 'Fred',
 'Funded',
 'Funding',
 'Generalized',
 'Germany-based',
 'Getting',
 'Guaranteed',
 'Having',
 'Heating',
 'Heightened',
 'Holding',
 'Housing',
 'Illumin

#### <span style="color:crimson">[연습😉3]</span> 계속 읽기 전에 위에 나열된 기호들의 의미를 파악하자.

<img src="https://github.com/kikim6114/nlp2026/blob/main/images/re_cheatsheet1.png?raw=1" width="600" style="margin-left: auto; margin-right: auto">
<p style="text-align: center;">와일드카드, 범위 및 클로저를 포함한 기본 정규 표현식 메타 문자</p>

## 2) 정규표현식의 응용

### Word Pieces 추출

In [65]:
word = 'supercalifragilisticexpialidocious'
re.findall(r'[aeiou]', word)

['u',
 'e',
 'a',
 'i',
 'a',
 'i',
 'i',
 'i',
 'e',
 'i',
 'a',
 'i',
 'o',
 'i',
 'o',
 'u']

In [ ]:
len(re.findall(r'[aeiou]', word))

16

In [ ]:
wsj = sorted(set(nltk.corpus.treebank.words()))
fd = nltk.FreqDist(vs for word in wsj
                      for vs in re.findall(r'[aeiou]{2,}', word))
fd.most_common(12)

[('io', 549),
 ('ea', 476),
 ('ie', 331),
 ('ou', 329),
 ('ai', 261),
 ('ia', 253),
 ('ee', 217),
 ('oo', 174),
 ('ua', 109),
 ('au', 106),
 ('ue', 105),
 ('ui', 95)]

**<span style="color:crimson">[연습😉4]</span>** W3C Date Time Format에서는 날짜가 2022-04-05 처럼 표현된다. 아래 파이썬 코드에서 `?`를 정규표현식으로 바꿔서 string '2022-04-05'를 정수 리스트 [2022, 4, 5] 가 되게 하라.
```\
[int(n) for n in re.findall(?, '2022-04-05')]
```

### Word Pieces를 사용한 추가 작업
- 영어 텍스트는 예측가능성이 높아서 단어 중간의 모음들을 생략해도 읽기가 용이한 특징이 있다
- declaration -> dclrtn, inalienable -> inlnble
- 첫 모음 또는 끝 모음은 그대로 둔다
- 첫 모음 순서, 끝 모음 순서, 그 다음에는 모든 자음 순서가 맞는 것을 찾는다.

In [ ]:
regexp = r'^[AEIOUaeiou]+|[AEIOUaeiou]+$|[^AEIOUaeiou]'

In [ ]:
def compress(word):
    pieces = re.findall(regexp, word)
    return ''.join(pieces)

In [ ]:
import nltk
nltk.download('udhr')

[nltk_data] Downloading package udhr to
[nltk_data]     C:\Users\kikim\AppData\Roaming\nltk_data...
[nltk_data]   Package udhr is already up-to-date!


True

In [ ]:
english_udhr = nltk.corpus.udhr.words('English-Latin1')
print(nltk.tokenwrap(compress(w) for w in english_udhr[:75]))

Unvrsl Dclrtn of Hmn Rghts Prmble Whrs rcgntn of the inhrnt dgnty and
of the eql and inlnble rghts of all mmbrs of the hmn fmly is the fndtn
of frdm , jstce and pce in the wrld , Whrs dsrgrd and cntmpt fr hmn
rghts hve rsltd in brbrs acts whch hve outrgd the cnscnce of mnknd ,
and the advnt of a wrld in whch hmn bngs shll enjy frdm of spch and


다음은 정규표현과 conditional frequency distribution을 결합한다. Rotokas어에서 ka나 si 같은 모든 자음-모음 sequence를 추출한다. 이것들 모두가 쌍이므로, conditional frequency distribution을 초기화하는데 사용할 수 있다. 그런 다음 각 쌍의 빈도를 표 형태로 출력한다.

In [68]:
import nltk
nltk.download('toolbox')

[nltk_data] Downloading package toolbox to /root/nltk_data...
[nltk_data]   Unzipping corpora/toolbox.zip.


True

In [69]:
rotokas_words = nltk.corpus.toolbox.words('rotokas.dic')
cvs = [cv for w in rotokas_words for cv in re.findall(r'[ptksvr][aeiou]', w)]
cfd = nltk.ConditionalFreqDist(cvs)
cfd.tabulate()

    a   e   i   o   u 
k 418 148  94 420 173 
p  83  31 105  34  51 
r 187  63  84  89  79 
s   0   0 100   2   1 
t  47   8   0 148  37 
v  93  27 105  48  49 


- 's'와 't' 행을 보면 이들이 부분적으로 "상보적 분포"를 갖는다는 것을 알 수 있다.
- 이들은 언어상 상이한 음소가 아니라는 증거이다.
- 따라서 Rotokas 알파벳에서 s를 빼버리고 t 다음에 i 가 나오면 t를 s로 발음하도록 하는 발음규칙을 만들수도 있을 것이다.
- 'su'는 빈도가 1인데 이것은 *kasuari*로 영어 'cassowary'에서 가져온 것이다.

추가적인 분석을 원한다면 주어진 consonant-vowel pair를 포함하는 단어 리스트를 빠르게 찾을 수 있도록 인덱스를 만드는 것이 편리할 것이다.

In [70]:
cv_word_pairs = [(cv, w) for w in rotokas_words
                         for cv in re.findall(r'[ptksvr][aeiou]', w)]
cv_index = nltk.Index(cv_word_pairs)
cv_index['su']

['kasuari']

In [71]:
cv_index['po']

['kaapo',
 'kaapopato',
 'kaipori',
 'kaiporipie',
 'kaiporivira',
 'kapo',
 'kapoa',
 'kapokao',
 'kapokapo',
 'kapokapo',
 'kapokapoa',
 'kapokapoa',
 'kapokapora',
 'kapokapora',
 'kapokaporo',
 'kapokaporo',
 'kapokari',
 'kapokarito',
 'kapokoa',
 'kapoo',
 'kapooto',
 'kapoovira',
 'kapopaa',
 'kaporo',
 'kaporo',
 'kaporopa',
 'kaporoto',
 'kapoto',
 'karokaropo',
 'karopo',
 'kepo',
 'kepoi',
 'keposi',
 'kepoto']

### 단어 어간 찾기
- 검색 엔진에 입력할 때 단어의 끝부분이 다른 것을 신경쓰지 않아도 되는 것은 stemming 기능 때문이다.
- 가장 간단한 방법으로 suffix처럼 보이는 것은 다 잘라버리는 방법을 사용해보자.
- 물론 NLTK에는 stemmer가 있다.

In [ ]:
def stem(word):
    for suffix in ['ing', 'ly', 'ed', 'ious', 'ies', 'ive', 'es', 's', 'ment']:
        if word.endswith(suffix):
            return word[:-len(suffix)]
    return word

- disjunction('OR'ing) of all suffixes
- disjunction의 범위를 한정하기 위해 괄호를 쓴다.

In [ ]:
re.findall(r'^.*(ing|ly|ed|ious|ies|ive|es|s|ment)$', 'processing')

['ing']

- matching이 된 단어 'processing'을 반환하지 않고 suffix를 반환하는 이유는 괄호 () 때문
- 단어를 출력하려면 `?:` 를 추가해야 한다.

In [ ]:
re.findall(r'^.*(?:ing|ly|ed|ious|ies|ive|es|s|ment)$', 'processing')

['processing']

- stem과 suffix 분리

In [ ]:
re.findall(r'^(.*)(ing|ly|ed|ious|ies|ive|es|s|ment)$', 'processing')

[('process', 'ing')]

*좋아 보이지만 아직 문제가 있다.*

In [ ]:
re.findall(r'^(.*)(ing|ly|ed|ious|ies|ive|es|s|ment)$', 'processes')

[('processe', 's')]

- 'es' 대신 's'를 찾았다.
- 첫번째 연산자가 'greedy'하여 '.\*' 부분이 가능한 많은 입력을 소비하려하기 때문
- `*` 연산자의 'non-greedy' 버전 `*?`을 사용하자.

In [ ]:
re.findall(r'^(.*?)(ing|ly|ed|ious|ies|ive|es|s|ment)$', 'processes')

[('process', 'es')]

두 번째 괄호를 option으로 만들면 empty suffix에서도 잘 작동한다.

In [ ]:
re.findall(r'^(.*?)(ing|ly|ed|ious|ies|ive|es|s|ment)?$', 'language')

[('language', '')]

아직도 문제가 많지만 그대로 진행하여, 전체 문서에 적용할 stemmer 함수를 만들자.

In [ ]:
def stem(word):
    regexp = r'^(.*?)(ing|ly|ed|ious|ies|ive|es|s|ment)?$'
    stem, suffix = re.findall(regexp, word)[0]
    return stem

In [ ]:
raw = """DENNIS: Listen, strange women lying in ponds distributing swords
is no basis for a system of government.  Supreme executive power derives from
a mandate from the masses, not from some farcical aquatic ceremony."""
tokens = word_tokenize(raw)
[stem(t) for t in tokens]

['DENNIS',
 ':',
 'Listen',
 ',',
 'strange',
 'women',
 'ly',
 'in',
 'pond',
 'distribut',
 'sword',
 'i',
 'no',
 'basi',
 'for',
 'a',
 'system',
 'of',
 'govern',
 '.',
 'Supreme',
 'execut',
 'power',
 'deriv',
 'from',
 'a',
 'mandate',
 'from',
 'the',
 'mass',
 ',',
 'not',
 'from',
 'some',
 'farcical',
 'aquatic',
 'ceremony',
 '.']

- *ponds*에서 *s*를 제거했으나, *is*와 *basis*에서도 제거했다.
- *distribut*나 *deriv* 같은 이상한 단어를 만들었다.

### 토큰화된 텍스트 검색
- `< >`: token boundary를 나타내며 그 사이의 공백은 무시한다.
- `(<.*>)`: single token, ()는 일치하는 단어(phrase 아니고)만 찾는다.

In [ ]:
import nltk
nltk.download('gutenberg')

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\kikim\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


True

In [ ]:
from nltk.corpus import gutenberg, nps_chat
moby = nltk.Text(gutenberg.words('melville-moby_dick.txt'))
moby.findall(r"<a> (<.*>) <man>")

monied; nervous; dangerous; white; white; white; pious; queer; good;
mature; white; Cape; great; wise; wise; butterless; white; fiendish;
pale; furious; better; certain; complete; dismasted; younger; brave;
brave; brave; brave


In [ ]:
chat = nltk.Text(nps_chat.words())

In [ ]:
chat.findall(r"<.*> <.*> <bro>")

you rule bro; telling you bro; u twizted bro


In [ ]:
chat.findall(r"<l.*>{3,}")

lol lol lol; lmao lol lol; lol lol lol; la la la la la; la la la; la
la la; lovely lol lol love; lol lol lol.; la la la; la la la


**<span style="color:crimson">[연습😉5]</span>** 패턴 p와 일치하는 모든 위치를 표시하기 위해 문자열 s에 주석을 추가하는 `nltk.re_show(p, s)`와 정규식 탐색을 위한 그래픽 인터페이스를 제공하는 nltk.app.nemo()를 사용하여 정규표현식의 패턴과 대입에 대한 이해를 통합하시오. 더 많은 연습을 위해 이 장의 끝에 있는 정규표현식 몇 가지를 연습해 보자.

- 조사하려는 언어학적 현상이 특정 단어들에 엮여 있으면 검색 패턴을 만들기 쉬워진다.
- "*x and other ys*"라는 표현은 대규모 코퍼스에서 hypernym을 찾기 쉽게 해준다.

In [ ]:
import nltk
nltk.download('brown')

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\kikim\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!


True

In [ ]:
from nltk.corpus import brown
hobbies_learned = nltk.Text(brown.words(categories=['hobbies', 'learned']))
hobbies_learned.findall(r"<\w*> <and> <other> <\w*s>")

speed and other activities; water and other liquids; tomb and other
landmarks; Statues and other monuments; pearls and other jewels;
charts and other items; roads and other features; figures and other
objects; military and other areas; demands and other factors;
abstracts and other compilations; iron and other metals


- 그러나 항상 false positive가 있음을 명심하자.
- 항상 false negative 문제도 있기 마련인데, 검색 패턴에 일치하는 것이 없다고 어떤 언어학적 현상이 코퍼스에 없다고 결론을 내리는 것은 위험하다. 적합한 패턴에 대해 더 신중히 했어야 했다.

## 3) 텍스트 토큰화를 위한 정규표현식

- 토큰화는 문자열을 쪼개어 언어 데이터를 구성하는 식별가능한 언어학적 단위들로 만드는 작업이다
- 지금까지 다룬 많은 말뭉치들이 이미 토큰화되어 있었으며, nltk도 약간의 토큰화 기능을 제공한다
- 정규표현식을 사용해 토큰화 시켜본다.

### 간단한 토큰화 방법
- 우선, 공백으로 나누기
- 텍스트: Alice's Adventures in Wonderland

In [ ]:
raw = """'When I'M a Duchess,' she said to herself, (not in a very hopeful tone
though), 'I won't have any pepper in my kitchen AT ALL. Soup does very
well without--Maybe it's always pepper that makes people hot-tempered,'..."""

In [ ]:
import re
print(re.split(r' ', raw))

["'When", "I'M", 'a', "Duchess,'", 'she', 'said', 'to', 'herself,', '(not', 'in', 'a', 'very', 'hopeful', 'tone\nthough),', "'I", "won't", 'have', 'any', 'pepper', 'in', 'my', 'kitchen', 'AT', 'ALL.', 'Soup', 'does', 'very\nwell', 'without--Maybe', "it's", 'always', 'pepper', 'that', 'makes', 'people', "hot-tempered,'..."]


- split() 함수를 사용하면 `\n`가 포함되므로 원하던 것이 아님
- space, tab, newline 모두 매치시켜야 한다

In [ ]:
print(re.split(r'[ \t\n]+', raw))

["'When", "I'M", 'a', "Duchess,'", 'she', 'said', 'to', 'herself,', '(not', 'in', 'a', 'very', 'hopeful', 'tone', 'though),', "'I", "won't", 'have', 'any', 'pepper', 'in', 'my', 'kitchen', 'AT', 'ALL.', 'Soup', 'does', 'very', 'well', 'without--Maybe', "it's", 'always', 'pepper', 'that', 'makes', 'people', "hot-tempered,'..."]


- '(not' 과 'herself,' 같은 토큰이 나왔다

In [ ]:
print(re.split(r'\W+', raw))

['', 'When', 'I', 'M', 'a', 'Duchess', 'she', 'said', 'to', 'herself', 'not', 'in', 'a', 'very', 'hopeful', 'tone', 'though', 'I', 'won', 't', 'have', 'any', 'pepper', 'in', 'my', 'kitchen', 'AT', 'ALL', 'Soup', 'does', 'very', 'well', 'without', 'Maybe', 'it', 's', 'always', 'pepper', 'that', 'makes', 'people', 'hot', 'tempered', '']


- 시작과 끝 부분에 빈 문자열이 나옴
- `\w+|\S\w*`: 임의의 문자열을 찾고 없으면 (non-whitespace 문자+단어문자열) 형태를 찾음
- 구둣점이 처음에 오면 문자열로 간주, 두 개이상의 구둣점이 처음에 오면 분리

In [ ]:
print(re.findall(r'\w+|\S\w*', raw))

["'When", 'I', "'M", 'a', 'Duchess', ',', "'", 'she', 'said', 'to', 'herself', ',', '(not', 'in', 'a', 'very', 'hopeful', 'tone', 'though', ')', ',', "'I", 'won', "'t", 'have', 'any', 'pepper', 'in', 'my', 'kitchen', 'AT', 'ALL', '.', 'Soup', 'does', 'very', 'well', 'without', '-', '-Maybe', 'it', "'s", 'always', 'pepper', 'that', 'makes', 'people', 'hot', '-tempered', ',', "'", '.', '.', '.']


- `w+`를 일반화시켜 단어 내부의 hyphens 와 apostrophes을 허용해보자
- `\w+([-']\w+)*`: 임의의 단어문자열+ hyphens 또는 apostrophes + 임의의 단어문자열
- 'hot-tempered' 나 'it's' 같은 단어가 포함될 것이다.
- `?`는 앞에서 논의한 이유로 포함시켜야 한다.

In [ ]:
print(re.findall(r"\w+(?:[-']\w+)*|'|[-.(]+|\S\w*", raw))

["'", 'When', "I'M", 'a', 'Duchess', ',', "'", 'she', 'said', 'to', 'herself', ',', '(', 'not', 'in', 'a', 'very', 'hopeful', 'tone', 'though', ')', ',', "'", 'I', "won't", 'have', 'any', 'pepper', 'in', 'my', 'kitchen', 'AT', 'ALL', '.', 'Soup', 'does', 'very', 'well', 'without', '--', 'Maybe', "it's", 'always', 'pepper', 'that', 'makes', 'people', 'hot-tempered', ',', "'", '...']


- `[-.(]+`: double hyphen, ellipsis 및 open parenthesis를 별도 토큰 처리한다.

<img src="https://github.com/kikim6114/nlp2026/blob/main/images/re_exp_symbols.png?raw=1" width="600" style="margin-left: auto; margin-right: auto">
<p style="text-align: center;">정규표현식 심볼</p>

### NLTK의 정규표현식 Tokenizer
- `nltk.regexp_tokenize()`: 괄호의 특별처리 등이 필요 없는 점 등 때문에 `re.findall()` 보다 능률적.
- verbose flag `(?x)`: 정규식 문자열 내의 공백 및 커멘트를 없앤다.
- verbose flag을 쓰면 공백을 지정하기 위해 '　'을 사용할 수 없게 되며, \s를 사용해야 함.

In [ ]:
text = 'That U.S.A. poster-print costs $12.40...'
pattern = r'''(?x)     # set flag to allow verbose regexps
    (?:[A-Z]\.)+       # abbreviations, e.g. U.S.A.
  | \w+(?:-\w+)*       # words with optional internal hyphens
  | \$?\d+(?:\.\d+)?%? # currency and percentages, e.g. $12.40, 82%
  | \.\.\.             # ellipsis
  | [][.,;"'?():-_`]   # these are separate tokens; includes ], [
'''
nltk.regexp_tokenize(text, pattern)

['That', 'U.S.A.', 'poster-print', 'costs', '$12.40', '...']

- `set(tokens).difference(wordlist)`: worldlist와 비교해서 wordlist에 없는 토큰을 반환하여 **토큰화 결과 비교**

### 토큰화 관련 추가 문제들
- 토큰화는 생각보다 훨씬 어렵다
- 한가지 해법으로 해결할 수 없으며, 문제 영역에 따라 결정을 해야 한다.
- 작성한 tokenizer의 출력을 고품질의 (gold-standard) 토큰들과 비교하기 위해 raw text를 검토하는 것이 바람직
- NLTK에는 raw Wall Street Journal text (nltk.corpus.treebank_raw.raw())와 토큰화된 버전(nltk.corpus.treebank.words())을 포함하여 Penn Treebank data 샘플이 포함되어 있다.
- *didn't* 같은 축약형은 do와 not 또는 do와 n't로 분리하는 것이 의미를 분석하기에 유리할 수 있다. 이를 위해 lookup table을 사용할 수 있다.

# 2.7   Segmentation
- Tokenization은 segmentation 문제의 더욱 일반화된 사례이다.
- 앞서 다룬 방법들과는 전혀 다른 기법들의 예를 살펴 본다.

## 1) Sentence Segmentation

In [ ]:
len(nltk.corpus.brown.words()) / len(nltk.corpus.brown.sents())

20.250994070456922

- 이 처리는 코퍼스가 문장 구분이 되어있다는 전제로 처리 가능했다.
- 코퍼스가 문자 스트림으로 되어 있는 경우가 흔하다.
- NLTK는 **Punkt sentence segmenter**([Kiss & Strunk, 2006](https://www.nltk.org/book/bibliography.html#kissstrunk2006))를 제공한다.

In [ ]:
import pprint
text = nltk.corpus.gutenberg.raw('chesterton-thursday.txt')
sents = nltk.sent_tokenize(text)
pprint.pprint(sents[79:89])

['"Nonsense!"',
 'said Gregory, who was very rational when anyone else\nattempted paradox.',
 '"Why do all the clerks and navvies in the\n'
 'railway trains look so sad and tired, so very sad and tired?',
 'I will\ntell you.',
 'It is because they know that the train is going right.',
 'It\n'
 'is because they know that whatever place they have taken a ticket\n'
 'for that place they will reach.',
 'It is because after they have\n'
 'passed Sloane Square they know that the next station must be\n'
 'Victoria, and nothing but Victoria.',
 'Oh, their wild rapture!',
 'oh,\n'
 'their eyes like stars and their souls again in Eden, if the next\n'
 'station were unaccountably Baker Street!"',
 '"It is you who are unpoetical," replied the poet Syme.']


- Sentence segmentation은 마침표(`.`) 때문에 특히 어렵다
- 문장 종료와 약어 표시 외에도 어떤 경우 약어와 종료를 겸하기도 함
- 또 다른 접근법은 Chapter 6의 Section 2 Further Examples of Supervised Classification을 참조하시오.

## 2) Word Segmentation

- 띄어쓰기가 없는 언어에서는 특히 토큰화가 어렵다.
- 중국어에서, 爱国人(ai4 "love" (verb), guo2 "country", ren2 "person")는 (爱国 + 人)(country-loving person) 또는 (爱 + 国人)(love country-person)으로 토큰화될 수 있다.
- 비슷한 문제가 음성 언어 처리에서도 일어난다. 듣는 사람은 연속적인 음성 흐름을 분할해야 한다.
- 단어를 미리 알고 있지 않은 경우 문제가 더 어려워진다.
- 단어 경계를 없앤 다음 예를 보자:
> (예)  
> a. doyouseethekitty  
> b. seethedoggy  
> c. doyoulikethekitty  
> d. likethedoggy  



- 각 글자에 대해 글자 다음에 word-break가 나타나는가 여부에 따라 1 또는 0을 표시해둔다
- 이 방법은 Chapter 7의 "Chunking"에서 집중적으로 사용된다.
- 가정: learner에게 utterance break가 주어진다(이는 실제 종종 *늘어진 멈춤*에 대응된다).

In [ ]:
text = "doyouseethekittyseethedoggydoyoulikethekittylikethedoggy"
seg1 = "0000000000000001000000000010000000000000000100000000000"  # 초기 segmentation
seg2 = "0100100100100001001001000010100100010010000100010010000"  # 목표 segmentation

In [ ]:
def segment(text, segs):
    words = []
    last = 0
    for i in range(len(segs)):
        if segs[i] == '1':
            words.append(text[last:i+1])
            last = i+1
    words.append(text[last:])
    return words

In [ ]:
text = "doyouseethekittyseethedoggydoyoulikethekittylikethedoggy"
seg1 = "0000000000000001000000000010000000000000000100000000000"
seg2 = "0100100100100001001001000010100100010010000100010010000"
segment(text, seg1)

['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']

In [ ]:
segment(text, seg2)

['do',
 'you',
 'see',
 'the',
 'kitty',
 'see',
 'the',
 'doggy',
 'do',
 'you',
 'like',
 'the',
 'kitty',
 'like',
 'the',
 'doggy']

- learner는 단어를 취득하여 내부 lexicon에 저장한다.
- [Brent (1995)](https://www.nltk.org/book/bibliography.html#brent1995)의 방법을 따라 **목적함수**를 만들 수 있다(그림 3.8)
- 목적함수 = 사전(lexicon) 크기(단어 내 글자 수 + 각 단어의 끝을 나타내는 별도의 구획 문자)와 lexicon에서 원래 텍스트를 복원하기 위해 필요한 정보량을 기반으로 하는 scoring function

<img src="https://github.com/kikim6114/nlp2026/blob/main/images/segmentation_objective.png?raw=1" width="600" style="margin-left: auto; margin-right: auto">
<p style="text-align: center;">세크멘테이션 목적함수</p>

In [ ]:
def evaluate(text, segs):
    words = segment(text, segs)
    text_size = len(words)
    lexicon_size = sum(len(word) + 1 for word in set(words))
    return text_size + lexicon_size

In [ ]:
text = "doyouseethekittyseethedoggydoyoulikethekittylikethedoggy"
seg1 = "0000000000000001000000000010000000000000000100000000000"
seg2 = "0100100100100001001001000010100100010010000100010010000"
seg3 = "0000100100000011001000000110000100010000001100010000001"
segment(text, seg3)

['doyou',
 'see',
 'thekitt',
 'y',
 'see',
 'thedogg',
 'y',
 'doyou',
 'like',
 'thekitt',
 'y',
 'like',
 'thedogg',
 'y']

In [ ]:
evaluate(text, seg3)

47

In [ ]:
evaluate(text, seg2)

48

In [ ]:
evaluate(text, seg1)

64

- 마지막 단계는 목적함수를 최소화시키는 0과 1의 패턴을 찾는 것이다.
- 최적화된 segmentation에 단어 *thekitty*가 나오는 이유는 이것을 더 분리하기에는 정보가 부족하기 때문

**Non-Deterministic Search Using Simulated Annealing:**  
- Phrase segmenation에만 적용
- "temperature"에 비례하여 0과 1을 무작위로 교란시킨다
- 매 반복마다 temperature는 내려가고 단어 경계의 교란은 줄어든다
- 알고리즘이 non-deterministic이므로 결과는 매번 달라질 수 있다

In [ ]:
from random import randint

def flip(segs, pos):
    return segs[:pos] + str(1-int(segs[pos])) + segs[pos+1:]

def flip_n(segs, n):
    for i in range(n):
        segs = flip(segs, randint(0, len(segs)-1))
    return segs

def anneal(text, segs, iterations, cooling_rate):
    temperature = float(len(segs))
    while temperature > 0.5:
        best_segs, best = segs, evaluate(text, segs)
        for i in range(iterations):
            guess = flip_n(segs, round(temperature))
            score = evaluate(text, guess)
            if score < best:
                best, best_segs = score, guess
        score, segs = best, best_segs
        temperature = temperature / cooling_rate
        print(evaluate(text, segs), segment(text, segs))
    print()
    return segs

In [ ]:
text = "doyouseethekittyseethedoggydoyoulikethekittylikethedoggy"
seg1 = "0000000000000001000000000010000000000000000100000000000"
anneal(text, seg1, 5000, 1.2)

64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
64 ['doyouseethekitty', 'seethedoggy', 'doyoulikethekitty', 'likethedoggy']
61 ['doyouseethekittys', 'ee', 'thedoggy', 'doyoulikethekit', 'ty', 'like', 'thedoggy']
59 ['doy', 'ouseet', 'hekitty', 's', 'ee', 'thedoggy', 'doyouliket', 'hekitty', 'like', 'thedoggy']
59 ['doy', 'ouseet', 'hekitty', 's', 'ee', 'thedoggy

'0000100100000001001000000010000100010000000100010000000'

- 데이터가 충분하다면 어느 정도의 정확도를 갖을 수 있다.
- 이 방법은 시각적인 단어 경계 표상이 없는 언어체계에 대한 토큰화에 적용될 수 있을 것이다.

# 2.8 정수 인코딩(Integer Encoding)

## 1) dictionary 사용하기

In [72]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [73]:
raw_text = "A barber is a person. a barber is good person. a barber is huge person. he Knew A Secret! The Secret He Kept is huge secret. Huge secret. His barber kept his word. a barber kept his word. His barber kept his secret. But keeping and keeping such a huge secret to himself was driving the barber crazy. the barber went up a huge mountain."

In [74]:
# 문장 분할
sentences = sent_tokenize(raw_text)
sentences

['A barber is a person.',
 'a barber is good person.',
 'a barber is huge person.',
 'he Knew A Secret!',
 'The Secret He Kept is huge secret.',
 'Huge secret.',
 'His barber kept his word.',
 'a barber kept his word.',
 'His barber kept his secret.',
 'But keeping and keeping such a huge secret to himself was driving the barber crazy.',
 'the barber went up a huge mountain.']

In [75]:
vocab = {}
preprocessed_sentences = []
stop_words = set(stopwords.words('english'))

for sentence in sentences:
    # 단어 토큰화
    tokenized_sentence = word_tokenize(sentence)
    result = []

    for word in tokenized_sentence:
        word = word.lower() # 모든 단어를 소문자화하여 단어의 개수를 줄인다.
        if word not in stop_words: # 단어 토큰화 된 결과에 대해서 불용어를 제거한다.
            if len(word) > 2: # 단어 길이가 2이하인 경우에 대하여 추가로 단어를 제거한다.
                result.append(word)
                if word not in vocab:
                    vocab[word] = 0
                vocab[word] += 1
    preprocessed_sentences.append(result)
print(preprocessed_sentences)

[['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]


In [ ]:
print(vocab)

{'barber': 8, 'person': 3, 'good': 1, 'huge': 5, 'knew': 1, 'secret': 6, 'kept': 4, 'word': 2, 'keeping': 2, 'driving': 1, 'crazy': 1, 'went': 1, 'mountain': 1}


In [ ]:
print(vocab["barber"]) # 'barber'라는 단어의 빈도수 출력

8


In [ ]:
vocab_sorted = sorted(vocab.items(), key = lambda x:x[1], reverse = True)
print(vocab_sorted)

[('barber', 8), ('secret', 6), ('huge', 5), ('kept', 4), ('person', 3), ('word', 2), ('keeping', 2), ('good', 1), ('knew', 1), ('driving', 1), ('crazy', 1), ('went', 1), ('mountain', 1)]


In [ ]:
word_to_index = {}
i = 0
for (word, frequency) in vocab_sorted :
    if frequency > 1 : # 빈도수가 작은 단어는 제외.
        i = i + 1
        word_to_index[word] = i

print(word_to_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'word': 6, 'keeping': 7}


In [ ]:
vocab_size = 5
words_frequency = [word for word, index in word_to_index.items() if index >= vocab_size + 1] # 인덱스가 5 초과인 단어 제거
for w in words_frequency:
    del word_to_index[w] # 해당 단어에 대한 인덱스 정보를 삭제
print(word_to_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5}


In [ ]:
word_to_index['OOV'] = len(word_to_index) + 1

In [ ]:
print(word_to_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'OOV': 6}


In [ ]:
encoded_sentences = []
for sentence in preprocessed_sentences:
    encoded_sentence = []
    for word in sentence:
        try:
            encoded_sentence.append(word_to_index[word])
        except KeyError:
            encoded_sentence.append(word_to_index['OOV'])
    encoded_sentences.append(encoded_sentence)
print(encoded_sentences)

[[1, 5], [1, 6, 5], [1, 3, 5], [6, 2], [2, 4, 3, 2], [3, 2], [1, 4, 6], [1, 4, 6], [1, 4, 2], [6, 6, 3, 2, 6, 1, 6], [1, 6, 3, 6]]


## 2) Counter 사용하기

In [ ]:
from collections import Counter

In [ ]:
preprocessed_sentences

[['barber', 'person'],
 ['barber', 'good', 'person'],
 ['barber', 'huge', 'person'],
 ['knew', 'secret'],
 ['secret', 'kept', 'huge', 'secret'],
 ['huge', 'secret'],
 ['barber', 'kept', 'word'],
 ['barber', 'kept', 'word'],
 ['barber', 'kept', 'secret'],
 ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'],
 ['barber', 'went', 'huge', 'mountain']]

In [ ]:
# words = np.hstack(preprocessed_sentences)으로도 수행 가능.
all_words_list = sum(preprocessed_sentences, [])
print(all_words_list)

['barber', 'person', 'barber', 'good', 'person', 'barber', 'huge', 'person', 'knew', 'secret', 'secret', 'kept', 'huge', 'secret', 'huge', 'secret', 'barber', 'kept', 'word', 'barber', 'kept', 'word', 'barber', 'kept', 'secret', 'keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy', 'barber', 'went', 'huge', 'mountain']


In [ ]:
# 파이썬의 Counter 모듈을 이용하여 단어의 빈도수 카운트
vocab = Counter(all_words_list)
print(vocab)

Counter({'barber': 8, 'secret': 6, 'huge': 5, 'kept': 4, 'person': 3, 'word': 2, 'keeping': 2, 'good': 1, 'knew': 1, 'driving': 1, 'crazy': 1, 'went': 1, 'mountain': 1})


In [ ]:
print(vocab["barber"]) # 'barber'라는 단어의 빈도수 출력

8


In [ ]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
print(vocab)

[('barber', 8), ('secret', 6), ('huge', 5), ('kept', 4), ('person', 3)]


In [ ]:
word_to_index = {}
i = 0
for (word, frequency) in vocab :
    i = i + 1
    word_to_index[word] = i

print(word_to_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5}


## 3) NLTK의 FreqDist 사용하기

In [ ]:
from nltk import FreqDist
import numpy as np

In [ ]:
vocab = FreqDist(np.hstack(preprocessed_sentences))

In [ ]:
print(vocab["barber"]) # 'barber'라는 단어의 빈도수 출력

8


In [ ]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
print(vocab)

[('barber', 8), ('secret', 6), ('huge', 5), ('kept', 4), ('person', 3)]


In [ ]:
word_to_index = {word[0] : index + 1 for index, word in enumerate(vocab)}
print(word_to_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5}
